# BIM LLM Fine-Tuning Notebook




In [4]:
import sys
print(sys.executable)


c:\Users\kreddy\Documents\Projects\BIMMeetGeneratorAgent\.venv-training\Scripts\python.exe


In [5]:
from __future__ import annotations

import json
import random
import sys
from pathlib import Path
from typing import Any

import torch
from datasets import load_dataset
from peft import LoraConfig, PeftModel, TaskType, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
    set_seed,
)

# Resolve the repository root when Jupyter starts from either the repo root
# or the NoteBooks/ directory.
cwd = Path.cwd().resolve()
if (cwd / "datasets").is_dir():
    REPO_ROOT = cwd
elif (cwd.parent / "datasets").is_dir():
    REPO_ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Could not find datasets/. Start Jupyter from the repository root "
        "or open this notebook from NoteBooks/."
    )

NOTEBOOKS_DIR = REPO_ROOT / "NoteBooks"
DATASETS_DIR = REPO_ROOT / "datasets"
PROCESSED_DIR = NOTEBOOKS_DIR / "processed"
OUTPUT_DIR = NOTEBOOKS_DIR / "artifacts" / "smollm2-360m-bim-lora"

print(f"Python: {sys.executable}")
print(f"Repository: {REPO_ROOT}")
print(f"Datasets: {DATASETS_DIR}")
print(f"Prepared data: {PROCESSED_DIR}")
print(f"Model output: {OUTPUT_DIR}")


Python: c:\Users\kreddy\Documents\Projects\BIMMeetGeneratorAgent\.venv-training\Scripts\python.exe
Repository: C:\Users\kreddy\Documents\Projects\BIMMeetGeneratorAgent
Datasets: C:\Users\kreddy\Documents\Projects\BIMMeetGeneratorAgent\datasets
Prepared data: C:\Users\kreddy\Documents\Projects\BIMMeetGeneratorAgent\NoteBooks\processed
Model output: C:\Users\kreddy\Documents\Projects\BIMMeetGeneratorAgent\NoteBooks\artifacts\smollm2-360m-bim-lora


In [6]:
SYSTEM_PROMPT = (
    "You convert natural-language BIM requests into valid hierarchical JSON. "
    "Return JSON only, with no Markdown or explanation. Preserve IDs, references, "
    "meters, and the right_handed_z_up coordinate system."
)

EXPECTED_KEYS = {
    "project",
    "site",
    "building",
    "storeys",
    "elements",
    "materials",
    "relationships",
    "metadata",
}

# Smoke-test defaults:
# - MAX_LEVEL=3 avoids oversized Level 4-10 JSON that hits the token limit
# - num_train_epochs=1 verifies the full pipeline quickly
MAX_LEVEL = 3

config = {
    "model_name": "HuggingFaceTB/SmolLM2-360M-Instruct",
    "seed": 42,
    "validation_fraction": 0.1,
    "max_seq_length": 8192,
    "num_train_epochs": 1,
    "learning_rate": 0.0002,
    "weight_decay": 0.01,
    "warmup_ratio": 0.05,
    "per_device_train_batch_size": 1,
    "per_device_eval_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "logging_steps": 5,
    "save_total_limit": 2,
    "lora_rank": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
}


def compact_json(value: Any) -> str:
    return json.dumps(value, ensure_ascii=False, separators=(",", ":"))


def messages_for_record(record: dict[str, Any]) -> list[dict[str, str]]:
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": record["input"]},
        {"role": "assistant", "content": compact_json(record["output"])},
    ]


def render_messages(tokenizer: Any, messages: list[dict[str, str]], generation: bool) -> str:
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=generation,
    )


print(f"Smoke-test MAX_LEVEL = {MAX_LEVEL}")
print(f"Epochs = {config['num_train_epochs']}")
print(f"Max sequence length = {config['max_seq_length']}")



Smoke-test MAX_LEVEL = 3
Epochs = 1
Max sequence length = 8192


## 3. Prepare train / validation data

Combines all 10 level JSONL files, then keeps only Levels `1..MAX_LEVEL` for the smoke test.

With `MAX_LEVEL = 3` this should produce about **135 train** and **15 validation** records.


In [7]:
def read_records(dataset_dir: Path) -> list[dict[str, Any]]:
    records: list[dict[str, Any]] = []
    files = sorted(dataset_dir.glob("level-*.jsonl"))
    if len(files) != 10:
        raise ValueError(f"Expected 10 level datasets, found {len(files)}")

    for path in files:
        level = int(path.name.split("-")[1])
        with path.open(encoding="utf-8") as handle:
            for line_number, line in enumerate(handle, start=1):
                record = json.loads(line)
                if set(record) != {"input", "output"}:
                    raise ValueError(f"{path.name}:{line_number} has an invalid record shape")
                records.append(
                    {
                        "id": f"level-{level:02d}-{line_number:03d}",
                        "level": level,
                        "messages": messages_for_record(record),
                    }
                )
    return records


def stratified_split(
    records: list[dict[str, Any]], validation_fraction: float, seed: int
) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    rng = random.Random(seed)
    train: list[dict[str, Any]] = []
    validation: list[dict[str, Any]] = []

    for level in range(1, 11):
        level_records = [record for record in records if record["level"] == level]
        rng.shuffle(level_records)
        validation_count = max(1, round(len(level_records) * validation_fraction))
        validation.extend(level_records[:validation_count])
        train.extend(level_records[validation_count:])

    rng.shuffle(train)
    rng.shuffle(validation)
    return train, validation


def write_jsonl(path: Path, records: list[dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        for record in records:
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")


records = read_records(DATASETS_DIR)
train_records, validation_records = stratified_split(
    records,
    float(config["validation_fraction"]),
    int(config["seed"]),
)

# Smoke test: keep only Levels 1..MAX_LEVEL (short examples, no truncation)
train_records = [r for r in train_records if r["level"] <= MAX_LEVEL]
validation_records = [r for r in validation_records if r["level"] <= MAX_LEVEL]

print(f"Smoke-test MAX_LEVEL = {MAX_LEVEL}")
print(f"Smoke-test train examples: {len(train_records)}")
print(f"Smoke-test validation examples: {len(validation_records)}")
print(
    "Levels kept:",
    sorted({r["level"] for r in train_records + validation_records}),
)

write_jsonl(PROCESSED_DIR / "train.jsonl", train_records)
write_jsonl(PROCESSED_DIR / "validation.jsonl", validation_records)

manifest = {
    "mode": "smoke_test",
    "max_level": MAX_LEVEL,
    "source_total_records": len(records),
    "train_records": len(train_records),
    "validation_records": len(validation_records),
    "levels": sorted({record["level"] for record in train_records + validation_records}),
    "seed": config["seed"],
    "validation_fraction": config["validation_fraction"],
}
with (PROCESSED_DIR / "manifest.json").open("w", encoding="utf-8") as handle:
    json.dump(manifest, handle, indent=2)

print(json.dumps(manifest, indent=2))
print(f"Wrote prepared data to {PROCESSED_DIR}")


Smoke-test MAX_LEVEL = 3
Smoke-test train examples: 135
Smoke-test validation examples: 15
Levels kept: [1, 2, 3]
{
  "mode": "smoke_test",
  "max_level": 3,
  "source_total_records": 500,
  "train_records": 135,
  "validation_records": 15,
  "levels": [
    1,
    2,
    3
  ],
  "seed": 42,
  "validation_fraction": 0.1
}
Wrote prepared data to C:\Users\kreddy\Documents\Projects\BIMMeetGeneratorAgent\NoteBooks\processed


## 4. Load model and attach LoRA


In [ ]:
set_seed(int(config["seed"]))

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available. Restart the kernel after installing the CUDA "
        "build of PyTorch in .venv-training."
    )

model = AutoModelForCausalLM.from_pretrained(
    config["model_name"],
    dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
)
model.config.use_cache = False
model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=int(config["lora_rank"]),
    lora_alpha=int(config["lora_alpha"]),
    lora_dropout=float(config["lora_dropout"]),
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
)
model = get_peft_model(model, lora_config)
model.enable_input_require_grads()
model.print_trainable_parameters()
print(f"Training device: {torch.cuda.get_device_name(0)}")


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 699.64it/s]


trainable params: 3,276,800 || all params: 365,097,920 || trainable%: 0.8975


## 5. Tokenize dataset

Only assistant JSON tokens are trained. System/user tokens are masked with `-100`.


In [9]:
tokenizer = AutoTokenizer.from_pretrained(config["model_name"])
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
    
dataset = load_dataset(
    "json",
    data_files={
        "train": str(PROCESSED_DIR / "train.jsonl"),
        "validation": str(PROCESSED_DIR / "validation.jsonl"),
    },
)
max_length = int(config["max_seq_length"])


def tokenize(example: dict) -> dict:
    messages = example["messages"]
    full_text = render_messages(tokenizer, messages, generation=False)
    prompt_text = render_messages(tokenizer, messages[:-1], generation=True)
    encoded = tokenizer(
        full_text,
        truncation=True,
        max_length=max_length,
        add_special_tokens=False,
    )
    prompt_ids = tokenizer(
        prompt_text,
        truncation=True,
        max_length=max_length,
        add_special_tokens=False,
    )["input_ids"]
    labels = list(encoded["input_ids"])
    prompt_length = min(len(prompt_ids), len(labels))
    labels[:prompt_length] = [-100] * prompt_length
    encoded["labels"] = labels
    return encoded


tokenized = dataset.map(
    tokenize,
    remove_columns=dataset["train"].column_names,
    desc="Tokenizing BIM examples",
)

truncated_train = sum(len(row["input_ids"]) >= max_length for row in tokenized["train"])
truncated_validation = sum(
    len(row["input_ids"]) >= max_length for row in tokenized["validation"]
)
print(f"Truncated train examples: {truncated_train}")
print(f"Truncated validation examples: {truncated_validation}")
if truncated_train or truncated_validation:
    raise ValueError(
        "Some examples reached max_seq_length and may have truncated JSON. "
        "For the first smoke test keep MAX_LEVEL = 3. Later you can raise "
        "MAX_LEVEL carefully or filter out over-long examples."
    )
print("No truncated examples. Safe to train.")


Tokenizing BIM examples: 100%|██████████| 15/15 [00:00<00:00, 351.79 examples/s]

Truncated train examples: 0
Truncated validation examples: 0
No truncated examples. Safe to train.


## 6. Train

Smoke-test settings: **1 epoch**, Levels **1–3 only**, on the laptop GPU in bfloat16.


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=float(config["num_train_epochs"]),
    learning_rate=float(config["learning_rate"]),
    weight_decay=float(config["weight_decay"]),
    warmup_ratio=float(config["warmup_ratio"]),
    per_device_train_batch_size=int(config["per_device_train_batch_size"]),
    per_device_eval_batch_size=int(config["per_device_eval_batch_size"]),
    gradient_accumulation_steps=int(config["gradient_accumulation_steps"]),
    logging_steps=int(config["logging_steps"]),
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=int(config["save_total_limit"]),
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    fp16=False,
    bf16=True,
    gradient_checkpointing=True,
    dataloader_pin_memory=True,
    use_cpu=False,
    seed=int(config["seed"]),
)

collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100,
    return_tensors="pt",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=collator,
)

result = trainer.train()
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))

with (OUTPUT_DIR / "train_metrics.json").open("w", encoding="utf-8") as handle:
    json.dump(result.metrics, handle, indent=2)

print(f"Saved LoRA adapter and tokenizer to {OUTPUT_DIR}")
result.metrics


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss
1,0.833773,0.945591


Saved LoRA adapter and tokenizer to C:\Users\kreddy\Documents\Projects\BIMMeetGeneratorAgent\NoteBooks\artifacts\smollm2-360m-bim-lora


{'train_runtime': 1906.2347,
 'train_samples_per_second': 0.071,
 'train_steps_per_second': 0.009,
 'total_flos': 209758973424000.0,
 'train_loss': 0.8628733228234684,
 'epoch': 1.0}

## 7. Inference

Generate hierarchical BIM JSON from a natural-language prompt.


In [10]:
set_seed(int(config["seed"]))

adapter_dir = OUTPUT_DIR
if not adapter_dir.exists():
    raise FileNotFoundError(
        f"Trained LoRA adapter not found at {adapter_dir}. "
        "Run the training cells first, or check OUTPUT_DIR."
    )

# Load tokenizer saved with the adapter
tokenizer = AutoTokenizer.from_pretrained(adapter_dir)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load base model + trained LoRA weights
base_model = AutoModelForCausalLM.from_pretrained(
    config["model_name"],
    dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
)
model = PeftModel.from_pretrained(base_model, adapter_dir)
model = model.to("cuda")
model.eval()

print(f"Loaded trained LoRA adapter from {adapter_dir}")
model.print_trainable_parameters()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 362.66it/s]


Loaded trained LoRA adapter from C:\Users\kreddy\Documents\Projects\BIMMeetGeneratorAgent\NoteBooks\artifacts\smollm2-360m-bim-lora
trainable params: 0 || all params: 365,097,920 || trainable%: 0.0000


In [11]:
def load_trained_model(cfg: dict):
    adapter_dir = OUTPUT_DIR
    if not adapter_dir.exists():
        raise FileNotFoundError(f"LoRA adapter not found: {adapter_dir}")

    tok = AutoTokenizer.from_pretrained(adapter_dir)
    base = AutoModelForCausalLM.from_pretrained(
        cfg["model_name"],
        dtype=torch.bfloat16,
        low_cpu_mem_usage=True,
    )
    peft_model = PeftModel.from_pretrained(base, adapter_dir)
    peft_model = peft_model.to("cuda")
    peft_model.eval()
    return peft_model, tok


def generate_with_model(model, tokenizer, prompt: str, max_new_tokens: int = 4096) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]
    text = render_messages(tokenizer, messages, generation=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated_ids = output[0, inputs["input_ids"].shape[1] :]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()


# Reuse the in-memory trained model if still available; otherwise reload from disk.
try:
    inference_model = model
    inference_tokenizer = tokenizer
    print("Using in-memory trained model")
except NameError:
    inference_model, inference_tokenizer = load_trained_model(config)
    print("Loaded trained model from disk")


Using in-memory trained model


In [12]:
prompt = (
    "Create a one-storey office with four walls, a slab, one door, and two rooms."
)

response = generate_with_model(inference_model, inference_tokenizer, prompt, max_new_tokens=4096)

try:
    parsed = json.loads(response)
    print(json.dumps(parsed, ensure_ascii=False, indent=2))
except json.JSONDecodeError:
    print("Model response was not valid JSON:")
    print(response)


Model response was not valid JSON:
{
  "store": {
    "name": "Office",
    "type": "store",
    "height": "10 meters",
    "width": "20 meters",
    "floor": "first",
    "floor_number": "1",
    "floor_type": "first",
    "floor_level": "first",
    "floor_level_number": "1",
    "floor_level_name": "first",
    "floor_level_number_name": "1",
    "floor_level_number_index": "1",
    "floor_level_index": "1",
    "floor_index": "1",
    "floor_index_name": "1",
    "floor_index_number": "1",
    "floor_index_number_name": "1",
    "floor_index_number_index": "1",
    "floor_index_number_index_name": "1",
    "floor_index_number_index_name_index": "1",
    "floor_index_number_index_name_index_name": "1",
    "floor_index_number_index_name_index_name_index": "1",
    "floor_index_number_index_name_index_name_index_name": "1",
    "floor_index_number_index_name_index_name_index_name_index": "1",
    "floor_index_number_index_name_index_name_index_name_index_name": "1",
    "floor_index_

## 8. Evaluate

Checks JSON validity, top-level structure, and exact match on a small validation sample.


In [13]:
EVAL_LIMIT = 5  # increase later; CPU inference is slow
MAX_NEW_TOKENS = 4096


def read_validation(path: Path, limit: int) -> list[dict]:
    records: list[dict] = []
    with path.open(encoding="utf-8") as handle:
        for line in handle:
            if len(records) >= limit:
                break
            records.append(json.loads(line))
    return records


eval_records = read_validation(PROCESSED_DIR / "validation.jsonl", EVAL_LIMIT)
if not eval_records:
    raise ValueError("No validation records found. Run the prepare-data cell first.")

valid_json = 0
valid_structure = 0
exact_matches = 0
results: list[dict] = []

for record in eval_records:
    user_prompt = record["messages"][1]["content"]
    expected = json.loads(record["messages"][2]["content"])
    response = generate_with_model(
        inference_model, inference_tokenizer, user_prompt, MAX_NEW_TOKENS
    )
    item = {"id": record["id"], "valid_json": False, "valid_structure": False}
    try:
        predicted = json.loads(response)
        item["valid_json"] = True
        valid_json += 1
        if isinstance(predicted, dict) and set(predicted) == EXPECTED_KEYS:
            item["valid_structure"] = True
            valid_structure += 1
        if predicted == expected:
            exact_matches += 1
    except json.JSONDecodeError as exc:
        item["error"] = str(exc)
    results.append(item)
    print(f"{record['id']}: valid_json={item['valid_json']} structure={item['valid_structure']}")

total = len(eval_records)
metrics = {
    "examples": total,
    "json_validity": valid_json / total,
    "top_level_structure_accuracy": valid_structure / total,
    "exact_match": exact_matches / total,
    "results": results,
}

eval_path = OUTPUT_DIR / "evaluation.json"
eval_path.parent.mkdir(parents=True, exist_ok=True)
with eval_path.open("w", encoding="utf-8") as handle:
    json.dump(metrics, handle, indent=2)

print(json.dumps(metrics, indent=2))
print(f"Saved evaluation metrics to {eval_path}")


level-03-023: valid_json=False structure=False
level-03-015: valid_json=True structure=False
level-03-019: valid_json=True structure=False
level-02-042: valid_json=True structure=False
level-01-026: valid_json=True structure=False
{
  "examples": 5,
  "json_validity": 0.8,
  "top_level_structure_accuracy": 0.0,
  "exact_match": 0.0,
  "results": [
    {
      "id": "level-03-023",
      "valid_json": false,
      "valid_structure": false,
      "error": "Expecting property name enclosed in double quotes: line 201 column 68 (char 11893)"
    },
    {
      "id": "level-03-015",
      "valid_json": true,
      "valid_structure": false
    },
    {
      "id": "level-03-019",
      "valid_json": true,
      "valid_structure": false
    },
    {
      "id": "level-02-042",
      "valid_json": true,
      "valid_structure": false
    },
    {
      "id": "level-01-026",
      "valid_json": true,
      "valid_structure": false
    }
  ]
}
Saved evaluation metrics to C:\Users\kreddy\Documents